
# Thử nghiệm các phương pháp phân vùng và phát hiện biên (Image Segmentation & Edge Detection)
Notebook này thử nghiệm một số thuật toán để tách cây khỏi ảnh nền, giúp quá trình trích xuất đặc trưng hiệu quả hơn.

Các phương pháp bao gồm:
1. **Phát hiện biên (Edge Detection)**: Canny Edge Detection
2. **Phân ngưỡng (Thresholding)**: Otsu's Thresholding trên kênh màu (Vd: Kênh H hoặc S trong không gian HSV)
3. **Phân cụm (Clustering)**: K-Means Clustering để phân nhóm màu sắc (Background vs Foreground)
4. **Watershed Algorithm**: Phân vùng dựa trên hình thái học và marker


In [ ]:

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
from glob import glob

# Đặt thư mục chứa ảnh
DATA_DIR = '../data/raw'
image_paths = glob(os.path.join(DATA_DIR, '*.jpg'))

# Lấy một vài ảnh mẫu để thử nghiệm
sample_paths = image_paths[:3] if len(image_paths) >= 3 else image_paths

def show_images(titles, images, rows=1, cols=3, figsize=(15, 5)):
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten()
    for i, (title, img) in enumerate(zip(titles, images)):
        if len(img.shape) == 2:
            axes[i].imshow(img, cmap='gray')
        else:
            axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[i].set_title(title)
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()
    
print(f"Loaded {len(sample_paths)} sample images for testing.")



## 1. Canny Edge Detection
Phương pháp phát hiện biên Canny giúp tìm ra đường viền của đối tượng trong ảnh.


In [ ]:

for path in sample_paths:
    img = cv2.imread(path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Làm mờ ảnh để giảm nhiễu
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # Canny Edge Detection
    edges = cv2.Canny(blurred, 50, 150)
    
    show_images(['Original', 'Grayscale + Blur', 'Canny Edges'], [img, blurred, edges])



## 2. Otsu's Thresholding (Trên không gian màu HSV)
Cây thường có màu xanh lục. Chúng ta có thể chuyển sang không gian màu HSV, lấy kênh Hue (màu sắc) hoặc Saturation (độ bão hòa) để áp dụng Otsu thresholding giúp tự động tìm ngưỡng phân chia.


In [ ]:

for path in sample_paths:
    img = cv2.imread(path)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    # Tách các kênh màu
    h, s, v = cv2.split(hsv)
    
    # Otsu thresholding trên kênh Saturation (vì nền thường nhạt màu hơn, cây đậm màu hơn)
    _, thresh_s = cv2.threshold(s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # Otsu trên kênh Hue
    _, thresh_h = cv2.threshold(h, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # Áp dụng mask lên ảnh gốc bằng mask của kênh S
    masked_img = cv2.bitwise_and(img, img, mask=thresh_s)
    
    show_images(['Original', 'Saturation Channel', 'Otsu (S-Channel) Mask', 'Masked Image'], 
                [img, s, thresh_s, masked_img], cols=4, figsize=(20, 5))



## 3. K-Means Clustering để phân vùng màu
Sử dụng K-Means để gom nhóm các pixel có màu sắc tương tự nhau. Bằng cách thiết lập K=2 hoặc K=3, ta có thể tách được phần thân/lá cây khỏi nền trời hoặc mặt đất.


In [ ]:

for path in sample_paths:
    img = cv2.imread(path)
    
    # Chuyển đổi ảnh thành mảng 2D các pixel
    Z = img.reshape((-1, 3))
    Z = np.float32(Z)
    
    # Tiêu chí dừng thuật toán
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
    K = 3 # Giả sử có 3 cụm: Lá cây, Nền trời, Đất/Cỏ
    
    # Áp dụng K-Means
    ret, label, center = cv2.kmeans(Z, K, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
    
    # Chuyển đổi trung tâm cụm thành số nguyên 8-bit
    center = np.uint8(center)
    res = center[label.flatten()]
    res2 = res.reshape((img.shape))
    
    # Hiển thị từng cụm
    cluster_masks = []
    for i in range(K):
        mask = (label == i).reshape(img.shape[:2]).astype(np.uint8) * 255
        cluster_masks.append(mask)
        
    titles = ['Original', 'K-Means (K=3)', 'Cluster 0', 'Cluster 1', 'Cluster 2']
    images = [img, res2] + cluster_masks
    
    show_images(titles, images, cols=5, figsize=(25, 5))



## 4. Phân vùng bằng thuật toán Watershed
Watershed coi ảnh như một bề mặt địa hình, với mức sáng tối đại diện cho độ cao. Sử dụng các "marker" được xác định từ trước để tránh hiện tượng phân vùng quá mức (over-segmentation).


In [ ]:

for path in sample_paths:
    img = cv2.imread(path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Otsu thresholding
    ret, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    # Noise removal
    kernel = np.ones((3,3), np.uint8)
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)
    
    # Chắc chắn vùng nền (sure background)
    sure_bg = cv2.dilate(opening, kernel, iterations=3)
    
    # Tìm vùng chắc chắn là foreground (sure foreground) bằng Distance Transform
    dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    ret, sure_fg = cv2.threshold(dist_transform, 0.7*dist_transform.max(), 255, 0)
    
    # Tìm vùng không xác định (Unknown region)
    sure_fg = np.uint8(sure_fg)
    unknown = cv2.subtract(sure_bg, sure_fg)
    
    # Gắn nhãn (Marker labelling)
    ret, markers = cv2.connectedComponents(sure_fg)
    
    # Cộng 1 cho tất cả các nhãn để nền chắc chắn không phải là 0 mà là 1
    markers = markers + 1
    
    # Đánh dấu vùng không xác định là 0
    markers[unknown == 255] = 0
    
    # Áp dụng Watershed
    markers = cv2.watershed(img, markers)
    
    # Tô màu biên bằng màu đỏ
    img_result = img.copy()
    img_result[markers == -1] = [0, 0, 255]
    
    show_images(['Original', 'Distance Transform', 'Markers', 'Watershed Result'], 
                [img, dist_transform, markers, img_result], cols=4, figsize=(20, 5))



## Kết luận
- **Canny Edge Detection**: Rất tốt để lấy được các chi tiết cạnh (cành, viền lá) nhưng dễ bị nhiễu do cây có cấu trúc phức tạp.
- **Otsu Thresholding (HSV/Saturation)**: Cách tiếp cận đơn giản và nhanh. Rất hiệu quả nếu nền và cây có độ chênh lệch rõ ràng về độ bão hòa (ví dụ trời xanh nhạt vs cây xanh đậm).
- **K-Means Clustering**: Phân cụm màu sắc làm rất tốt việc gom nhóm, đặc biệt khi các ảnh có màu sắc tách biệt (lá cây, bầu trời, đất). Có thể dùng để tạo mask rất tốt.
- **Watershed**: Hoạt động tốt nếu chúng ta có thể xác định marker tốt (sure background, sure foreground).
